# Laya HL Backtest: finetuned vs base on the recorded tape

Runs the **actual trained Laya checkpoints** through the offline backtest harness.
`scripts/backtest.py` on kinan only ever tested a 16-feature `LogisticRegression` placeholder;
this notebook replaces that predictor with the transformer and reuses the identical
trade/PnL math, cost hurdle, and chronological split.

**Trade rule (byte-identical to `scripts/backtest.py:236-260`):**
- `intent == open` AND `bias` direction -> take the trade
- PnL per trade = `direction * fwd_bps - hurdle_bps`
- `hurdle = max(spread + 2*fee, 12.0)` bps (same `ADVERSE_SELECTION_BPS` floor)

**Inputs:**
- `kinan21/laya-hl-trial-v2` (the relabeled 5-min dataset; carries `state` text, `questions`, `meta.fwd_bps`, `meta.hurdle_bps`)
- `kinan21/laya-hl-finetune-multilingual-v2` (kernel output -> the finetuned checkpoint at `laya_hl_small/`)

Breakeven win rates at the recorded hurdle (12.7 bps), measured on this tape:
- 5 min: 68.4% | 15 min: 63.1% | 30 min: 60.0%

Forward direction is a coin flip at these horizons (48.7% / 50.0% / 52.1% up), so the
honest expectation is that the model does NOT clear the bar. We run it to get the number.

In [ ]:
import json, os, sys, time, statistics
import numpy as np

DATA = None
for cand in ["/kaggle/input/laya-hl-trial-v2",
             "/kaggle/input/datasets/kinan21/laya-hl-trial-v2"]:
    if os.path.exists(os.path.join(cand, "train.jsonl")):
        DATA = cand; break
assert DATA, "dataset kinan21/laya-hl-trial-v2 not mounted"

# The finetuned checkpoint is uploaded as a private dataset. The parent kernel's
# output could NOT be attached (Kaggle rejected the kernel_sources reference),
# so we accept either the unpacked dir or the tarball and unpack it ourselves.
FT_CANDS = [
    "/kaggle/input/laya-hl-finetuned-ckpt-v2/laya_hl_small",
    "/kaggle/input/laya-hl-finetune-multilingual-v2/laya_hl_small",
]
FT = next((c for c in FT_CANDS if os.path.isdir(c)), None)
if FT is None:
    import tarfile
    for c in ["/kaggle/input/laya-hl-finetuned-ckpt-v2/laya_hl_small.tar.gz"]:
        if os.path.exists(c):
            with tarfile.open(c) as tf: tf.extractall("/kaggle/working")
            FT = "/kaggle/working/laya_hl_small"
            break
assert FT and os.path.exists(os.path.join(FT, "rl_agent_config.json")), \
    f"finetuned checkpoint not found; tried {FT_CANDS}"
print("DATA:", DATA)
print("finetuned dir:", FT)

In [ ]:
!pip install -q -U "laya>=0.3.21" "transformers>=4.48.0" safetensors huggingface_hub

In [ ]:
import laya
print("laya", getattr(laya, "__version__", "ok"))

# The base checkpoint is downloaded from HF (internet is on for this kernel).
from huggingface_hub import snapshot_download
BASE_ID = "convaiinnovations/laya-multilingual"
print("downloading base model", BASE_ID, "...")
base_dir = snapshot_download(BASE_ID)
# _fix_tokenizer_config is NOT in laya's public __all__; import from the submodule.
from laya.agent import _fix_tokenizer_config
if os.path.isdir(os.path.join(base_dir, "tokenizer")):
    _fix_tokenizer_config(base_dir)
print("base dir:", base_dir)

In [ ]:
def load_rows(path, limit=None):
    rows = []
    with open(path) as f:
        for i, line in enumerate(f):
            if limit and i >= limit: break
            if line.strip():
                rows.append(json.loads(line))
    return rows

# The dataset is 5-min labelled (median|fwd| ~10.9bps). We backtest it as-is:
# the trade math uses meta.fwd_bps + meta.hurdle_bps directly, so the horizon is
# whatever the labels were built at -- no relabel happens here.
val = load_rows(os.path.join(DATA, "val.jsonl"))
tr  = load_rows(os.path.join(DATA, "train.jsonl"))
print(f"train: {len(tr)}  val: {len(val)}")
print("val day(s):", sorted({r['meta']['day'] for r in val}))

In [ ]:
TAKER_FEE_BPS = 0.35
ADVERSE_SELECTION_BPS = 12.0

def gold_label(gold_q, keys):
    """Same derivation as the training cell: argmax over probabilities."""
    p = gold_q["probabilities"]
    return max(keys, key=lambda k: p.get(k, 0.0))

def run_model(agent, rows, tag, want_pnl=False):
    """Score an agent. Returns metrics + optional per-trade PnL records.

    Trade rule mirrors scripts/backtest.py exactly:
      intent == 'open' -> trade, direction from bias.
      pnl = direction * fwd_bps - hurdle_bps
    """
    acc, brier = {}, {}
    trades, wins, net_bps = 0, 0, 0.0
    pnl = []
    lat = []
    for i, row in enumerate(rows):
        state = row["state"] if isinstance(row["state"], str) else json.dumps(row["state"])
        qs = row["questions"]
        gold = row["gold"]
        t0 = time.perf_counter()
        res = agent.predict(state, qs)
        lat.append((time.perf_counter() - t0) * 1000)
        ans = res["answers"]
        for qid, qdef in qs.items():
            if qid not in gold: continue
            keys = list(qdef["criteria"].keys())
            pred = ans[qid]["choice"]
            lab = gold_label(gold[qid], keys)
            acc.setdefault(qid, []).append(float(pred == lab))
            p = np.array([ans[qid]["probabilities"].get(k, 1e-6) for k in keys], dtype=np.float64)
            p = p / p.sum()
            g = np.zeros(len(keys)); g[keys.index(lab)] = 1.0
            brier.setdefault(qid, []).append(float(((p - g) ** 2).sum()))
        # ---- trade decision (backtest.py semantics) ----
        intent_pred = ans.get("intent", {}).get("choice", "hold")
        if intent_pred == "open":
            bias_pred = ans.get("bias", {}).get("choice", "long")
            direction = 1.0 if bias_pred == "long" else -1.0
            fwd = row["meta"]["fwd_bps"]
            hurdle = max(row["meta"].get("hurdle_bps") or 0.0,
                         ADVERSE_SELECTION_BPS + 2 * TAKER_FEE_BPS)
            tp = direction * fwd - hurdle
            trades += 1; net_bps += tp
            if tp > 0: wins += 1
            if want_pnl:
                pnl.append({**{k: row["meta"][k] for k in ("coin","ts","day")},
                            "direction": direction, "fwd_bps": fwd,
                            "hurdle_bps": hurdle, "trade_pnl_bps": tp})
        if i % 500 == 0:
            print(f"  ..{i}/{len(rows)}", flush=True)
    m = {q: float(np.mean(v)) for q, v in acc.items()}
    m["overall_acc"] = float(np.mean([v for vs in acc.values() for v in vs]))
    m["overall_brier"] = float(np.mean([v for vs in brier.values() for v in vs]))
    m["mean_latency_ms"] = float(np.mean(lat))
    m["trades"] = trades
    m["net_bps"] = net_bps
    m["mean_trade_bps"] = (net_bps / trades) if trades else 0.0
    m["win_rate"] = (wins / trades) if trades else 0.0
    print(f"[{tag}] " + " | ".join(f"{k}={v:.4f}" for k, v in m.items() if k not in ("trades","net_bps"))
          + f" | trades={trades} net={net_bps:.1f}", flush=True)
    return (m, pnl) if want_pnl else m

In [ ]:
# Majority-class reference on the SAME rows, so we never confuse
# "the model beat nothing" with "the model beat something".
from collections import Counter
def majority_ref(rows, split_name):
    per = {}
    for r in rows:
        for qid, qdef in r["questions"].items():
            if qid not in r["gold"]: continue
            keys = list(qdef["criteria"].keys())
            per.setdefault(qid, []).append(gold_label(r["gold"][qid], keys))
    out = {q: Counter(v).most_common(1)[0][1] / len(v) for q, v in per.items()}
    out["overall_acc"] = float(np.mean(list(out.values())))
    # always-long, no filter -- the pure directional expectancy
    n = len(rows); cost = ADVERSE_SELECTION_BPS + 2 * TAKER_FEE_BPS
    al = [r["meta"]["fwd_bps"] - cost for r in rows]
    out["always_long_mean_bps"] = float(np.mean(al))
    out["n"] = n
    print(f"[{split_name} MAJORITY] " + " | ".join(f"{k}={v:.4f}" for k, v in out.items()))
    return out

In [ ]:
maj_val = majority_ref(val, "VAL")
maj_tr  = majority_ref(tr,  "TRAIN")

In [ ]:
print("== BASE (pretrained laya-multilingual) ==")
agent_b = laya.Agent(base_dir, device="cuda")
m_base_tr = run_model(agent_b, tr,  "base/train")
m_base    = run_model(agent_b, val, "base/val",   want_pnl=True)
del agent_b
import torch; torch.cuda.empty_cache()

In [ ]:
print("== FINETUNED ==")
agent_f = laya.Agent(FT, device="cuda")
m_ft_tr = run_model(agent_f, tr,  "ft/train")
m_ft, pnl_ft = run_model(agent_f, val, "ft/val", want_pnl=True)
del agent_f
torch.cuda.empty_cache()

In [ ]:
print("\n" + "=" * 78)
print("HEAD-TO-HEAD (val = held-out 2026-09-30)")
print("=" * 78)
rows_show = [
    ("bias acc",       m_base["bias"],            m_ft["bias"],            maj_val["bias"]),
    ("intent acc",     m_base["intent"],          m_ft["intent"],          maj_val["intent"]),
    ("leverage acc",   m_base["leverage"],        m_ft["leverage"],        maj_val["leverage"]),
    ("OVERALL acc",    m_base["overall_acc"],     m_ft["overall_acc"],     maj_val["overall_acc"]),
    ("OVERALL brier",  m_base["overall_brier"],   m_ft["overall_brier"],   None),
]
print(f"{'metric':16} {'base':>9} {'finetuned':>11} {'majority':>9} {'ft-base':>9}")
for name, b, f_, m in rows_show:
    d = f"{f_-b:+9.4f}" if b is not None else ""
    ms = f"{m:9.4f}" if m is not None else "        -"
    print(f"{name:16} {b:9.4f} {f_:11.4f} {ms} {d}")
print()
print(f"{'trades':16} {m_base['trades']:9d} {m_ft['trades']:11d}")
print(f"{'net bps':16} {m_base['net_bps']:9.1f} {m_ft['net_bps']:11.1f}")
print(f"{'mean bps/trade':16} {m_base['mean_trade_bps']:9.2f} {m_ft['mean_trade_bps']:11.2f}   (always-long ref {maj_val['always_long_mean_bps']:+.2f})")
print(f"{'win rate':16} {m_base['win_rate']:9.4f} {m_ft['win_rate']:11.4f}")
print(f"{'latency ms':16} {m_base['mean_latency_ms']:9.0f} {m_ft['mean_latency_ms']:11.0f}")

In [ ]:
# Per-coin breakdown of the finetuned PnL, same shape as backtest.py's table.
from collections import defaultdict
by_coin = defaultdict(lambda: [0, 0.0, 0])
for t in pnl_ft:
    c = by_coin[t["coin"]]; c[0] += 1; c[1] += t["trade_pnl_bps"]
    if t["trade_pnl_bps"] > 0: c[2] += 1
print(f"{'Coin':6} {'Trades':>7} {'Net(bps)':>10} {'WR%':>6}")
for coin in sorted(by_coin):
    n, net, w = by_coin[coin]
    print(f"{coin:6} {n:7d} {net:10.1f} {100*w/n:6.1f}")

In [ ]:
report = {
    "baseline": m_base, "finetuned": m_ft,
    "baseline_train": m_base_tr, "finetuned_train": m_ft_tr,
    "majority_val": maj_val, "majority_train": maj_tr,
    "breakeven_note": "win rate needed at 12.7bps hurdle: 68.4% (5m) / 63.1% (15m) / 60.0% (30m); this dataset is 5-min labelled",
    "val_cases": len(val),
}
with open("/kaggle/working/backtest_report.json", "w") as f:
    json.dump(report, f, indent=2)
with open("/kaggle/working/backtest_pnl_ft.jsonl", "w") as f:
    for t in pnl_ft:
        f.write(json.dumps(t) + "\n")
print("saved /kaggle/working/backtest_report.json + backtest_pnl_ft.jsonl")